In [0]:
%python
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
%python
def read_file(csv_path):
    return spark.read.csv(csv_path,header=True,inferSchema=True)

In [0]:
%python 

def changeType(df):
    return df.withColumn('Total Spent',F.col('Total Spent').try_cast('double'))\
             .withColumn('Quantity',F.col('Quantity').try_cast('int'))\
             .withColumn('Price Per Unit',F.col('Price Per Unit').try_cast('double'))\
             .withColumn('Transaction Date',F.to_date('Transaction Date' , 'yyyy-MM-dd'))

In [0]:
%python
def missingValuePerColumn(df):
    return df.select([
    F.count(F.when(F.col(c).isNull() , c)).alias(c) 
    for c in df.columns
])

In [0]:
%python
def mostAppearValue(colName,df):
    return df.groupby(F.col(colName)).count()

In [0]:
%python
def foundValues(df):
    return (df
        .withColumn('calc_total', F.col('Quantity') * F.col('Price Per Unit'))
        .withColumn('calc_qty', F.col('Total Spent') / F.col('Price Per Unit'))
        .withColumn('calc_price', F.col('Total Spent') / F.col('Quantity'))
        .withColumn('Total Spent', F.coalesce(F.col('Total Spent'), F.col('calc_total')))
        .withColumn('Quantity', F.coalesce(F.col('Quantity'), F.col('calc_qty')))
        .withColumn('Price Per Unit', F.coalesce(F.col('Price Per Unit'), F.col('calc_price')))
        .drop('calc_total', 'calc_qty', 'calc_price')
    )

In [0]:
%python
def findMissingValue(df):
    ref_price_df = df.groupBy('Item').agg(F.avg('Price Per Unit')).alias('ref_price')
    joined_df = df.join(ref_price_df , how="left",on="Item")
    return joined_df.withColumn('Price Per Unit',F.coalesce(F.col('Price Per Unit'), F.col('ref_price'))).drop('ref_price')

In [0]:
%python
def standardizeMissing(df):
    junk_values = ['ERROR', 'UNKNOWN', '','NA']
    cols_to_clean = [c for c in df.columns if c != 'Transaction ID']
    return df.na.replace(junk_values, None, subset=cols_to_clean)

In [0]:
%python
def fillWithItemAverage(df, colName):
    w = Window.partitionBy('Item')
    return df.withColumn(
        colName,
        F.coalesce(F.col(colName), F.avg(F.col(colName)).over(w))
    )

In [0]:
%python
def cleaningItemVal(df):
    item_lookup = spark.createDataFrame([
        (1.0, 'Cookie'),
        (1.5, 'Tea'),
        (2.0, 'Coffee'),
        (5.0, 'Salad'),
    ], ['Price Per Unit', 'lookup_item'])

    df = df.join(F.broadcast(item_lookup), on='Price Per Unit', how='left')
    return df.withColumn('Item', F.coalesce(F.col('Item'), F.col('lookup_item'))).drop('lookup_item')

In [0]:
%python
def findPriceWithItem(df):
    return (df
        .groupBy('Price Per Unit')
        .agg(F.collect_set('Item').alias('items'))
        .orderBy('Price Per Unit')
    )


In [0]:
%python
def cleanTransactionDate(df):
    return df.withColumn('day_type',
                F.when(F.col('Transaction Date').isNull() , 'Unknow' )
                 .when(F.dayofweek('Transaction Date').isin(1,7),'Weekend')
                 .otherwise('Weekday')
    )

In [0]:
%python
def fillUnknownPayment_Location(df):
    return df.withColumn('Payment Method',F.coalesce(F.col('Payment Method'), F.lit('--UNKNOW--')))\
             .withColumn('Location',F.coalesce(F.col('Location'), F.lit('--UNKNOW--')))

In [0]:
%python 
#KPI 1. Revenue by Product
def revByProduct(df):
    return (df.filter(F.col('Item') != '--UNKNOW--')
            .groupBy("Item")
            .agg(F.round(F.sum('Total Spent'),2).alias('Revenue'))
            .orderBy('Revenue',ascending=False))

In [0]:
%python 
#KPI 2. Revenue by Location
def revByLocation(df):
    return (df.filter(F.col('Location') != '--UNKNOW--')
            .groupBy("Location")
            .agg(F.round(F.sum('Total Spent'),2).alias('Revenue'))
            .orderBy('Revenue',ascending=False))

In [0]:
%python
# KPI 3. Revenue by Payment Method
def revByPaymentMethod(df):
    return (df.filter(F.col('Payment Method') != '--UNKNOW--')
            .groupBy("Payment Method")
            .agg(F.round(F.sum('Total Spent'),2).alias('Revenue'))
            .orderBy('Revenue',ascending=False))

In [0]:
%python
#KPI 4. Weekend vs Weekday Sales
def weekendVsweekday(df):
    return (df.groupBy('day_type')
            .agg(F.round(F.sum('Total Spent'),2).alias('Total Sales'))
            .orderBy(F.col('Total Sales') , ascending=False)
    )

In [0]:
%python
# KPI 5 . Peek Sale Day
def peekSaleDay(df):
    return (df.filter(F.col('Transaction Date').isNotNull())
            .groupBy('Transaction Date')
            .agg(F.round(F.sum('Total Spent'),2).alias('Revenue'))
            .orderBy(F.col('Revenue'),ascending = False)
            .limit(1)
    )

In [0]:

%python
# KPI 6 Best Selling Item (by Quantity)
def bestSellingItemByQunatity(df):
    return (df.filter(F.col('Item') != '--UNKNOW--')
            .groupBy("Item")
            .agg(F.round(F.sum('Quantity'),2).alias('Highest Selling'))
            .orderBy(F.col('Highest Selling').desc())
            .limit(1))

In [0]:
%python 
# KPI 7 Product Revenue Contribution (%)
def productRevContri(df):
    total_rev = df.agg(F.sum('Total Spent')).first()[0]
    return (df.groupBy('Item')
            .agg(F.round(F.sum('Total Spent') * 100 / total_rev,2).alias('Percentage'))
            .orderBy(F.col('Percentage').desc())
    )


In [0]:
%python 
# KPI 8. Transaction Success Rate
def transactionSuccessRate(df,placeholder = '--UNKNOW--'):
    total = df.count()
    isinCol = ['Payment Method','day_type','Location']
    successful = df.filter(
        ~F.array_contains(
            F.array(*[F.col(c).isNull() | (F.col(c) == placeholder) for c in df.columns if c in isinCol]),
            True
        )
    ).count()
    return round(successful * 100 / total,2)


In [0]:
%python
# KPI 9 Missing data rate
def missingDataRate(df):
    total_cell = df.count()*len(df.columns)
    missingNullCellwiseTotal = sum(missingValuePerColumn(df).collect()[0])
    return missingNullCellwiseTotal * 100 / total_cell


In [0]:
%python
# KPI 10 Invalid Date Percentage
def invalidDatePercentage(df):
    total_count = df.count()
    invalid_count = df.filter(F.col('Transaction Date').isNull()).count()
    return round(invalid_count * 100 / total_count,2)

In [0]:
%python
df = read_file('/Volumes/dbacademy/default/dirty_cafe_volume')
df = standardizeMissing(df)
df = changeType(df)
df = foundValues(df)
df = fillWithItemAverage(df,'Price Per Unit')
df = foundValues(df)

helper = df

deletedQTnull =  df.filter(F.col('Quantity').isNull() & F.col('Total Spent').isNull())
df = df.filter(F.col('Quantity').isNotNull() & F.col('Total Spent').isNotNull())

df = cleaningItemVal(df)
deletedItemNull = df.filter(F.col('Item').isNull())
df = df.filter(F.col('Item').isNotNull())

df = cleanTransactionDate(df)

df = fillUnknownPayment_Location(df)

display(revByProduct(df)) # KPI 1
display(revByLocation(df)) # KPI 2
display(revByPaymentMethod(df)) # KPI 3
display(weekendVsweekday(df)) # KPI 4
display(peekSaleDay(df)) # KPI 5
display(bestSellingItemByQunatity(df)) # KPI 6
display(productRevContri(df)) # KPI 7
print('Transaction Success Rate: ',transactionSuccessRate(df)) # KPI 8
print('Missing Data Rate: ',missingDataRate(helper)) # KPI 9
print('Invalid Date Percentage : ',invalidDatePercentage(df)) # KPI 10